# EPA ECHO — Iowa NPDES Facility Download

Downloads facility metadata for all Iowa NPDES permit holders from EPA's
Enforcement and Compliance History Online (ECHO) system.

ECHO facility data enriches the existing NPDES discharge monitoring records
(`NPDES_DMRS_FY*.csv`) with information not present in the DMR files:
facility type code, SIC/NAICS industry classification (including whether a
facility is a sewage treatment works), and geocoded location.

**Join key**: `npdes_id` in the outputs matches `EXTERNAL_PERMIT_NMBR` in the DMR files.

**Source**: ECHO bulk download ZIP at `https://echo.epa.gov/files/echodownloads/npdes_downloads.zip`  
Only the three relevant files are streamed from the ZIP using HTTP range requests —
the full 341 MB archive is never written to disk.

**Outputs**
- `data/tabular/01_raw/npdes/echo-facilities-iowa.csv` — 2,216 Iowa NPDES facilities  
- `data/tabular/01_raw/npdes/echo-naics-iowa.csv` — NAICS codes per permit  
- `data/tabular/01_raw/npdes/echo-sics-iowa.csv` — SIC codes per permit

In [ ]:
import csv
import io
import struct
import urllib.request
import zlib

import pandas as pd
from pathlib import Path

RAW_DIR = Path('../../data/tabular/01_raw/npdes')
RAW_DIR.mkdir(parents=True, exist_ok=True)

ZIP_URL = 'https://echo.epa.gov/files/echodownloads/npdes_downloads.zip'

# Pre-computed from ZIP central directory (run once; stable unless EPA repacks the file)
# If the download fails with a range error, re-run _parse_zip_directory() below.
FILES = {
    # name: (local_header_offset, compressed_size_bytes)
    'ICIS_FACILITIES.csv': (0,          69_473_500),
    'NPDES_NAICS.csv':     (203189521,   1_475_412),
    'NPDES_SICS.csv':      (204665933,   3_628_073),
}

In [ ]:
def _local_data_start(local_hdr_offset: int) -> int:
    """Return the byte offset where compressed data begins inside a ZIP local entry."""
    req = urllib.request.Request(
        ZIP_URL,
        headers={'Range': f'bytes={local_hdr_offset}-{local_hdr_offset + 99}'},
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        hdr = r.read()
    fn_len = struct.unpack_from('<H', hdr, 26)[0]
    ex_len = struct.unpack_from('<H', hdr, 28)[0]
    return local_hdr_offset + 30 + fn_len + ex_len


def stream_csv(filename: str) -> str:
    """Stream-decompress one CSV from the ECHO bulk ZIP without saving the archive."""
    local_offset, comp_size = FILES[filename]
    data_start = _local_data_start(local_offset)
    print(f'  Fetching {filename} ({comp_size / 1e6:.1f} MB compressed)...', flush=True)
    req = urllib.request.Request(
        ZIP_URL,
        headers={'Range': f'bytes={data_start}-{data_start + comp_size - 1}'},
    )
    with urllib.request.urlopen(req, timeout=300) as r:
        compressed = r.read()
    raw = zlib.decompressobj(-15).decompress(compressed)   # raw DEFLATE
    return raw.decode('utf-8', errors='replace')

## 1. ICIS Facility metadata

In [ ]:
text = stream_csv('ICIS_FACILITIES.csv')
reader = csv.DictReader(io.StringIO(text))
rows = [r for r in reader if r.get('STATE_CODE', '').strip().strip('"') == 'IA']

df_fac = pd.DataFrame(rows)
df_fac.columns = [c.strip('"') for c in df_fac.columns]

col_map = {
    'ICIS_FACILITY_INTEREST_ID': 'facility_interest_id',
    'NPDES_ID':                  'npdes_id',
    'FACILITY_UIN':              'facility_uin',
    'FACILITY_TYPE_CODE':        'facility_type_code',
    'FACILITY_NAME':             'facility_name',
    'LOCATION_ADDRESS':          'address',
    'CITY':                      'city',
    'COUNTY_CODE':               'county_fips',
    'ZIP':                       'zip',
    'GEOCODE_LATITUDE':          'latitude',
    'GEOCODE_LONGITUDE':         'longitude',
    'IMPAIRED_WATERS':           'impaired_waters',
}
df_fac = df_fac[[c for c in col_map if c in df_fac.columns]].rename(columns=col_map)
for col in ('latitude', 'longitude'):
    df_fac[col] = pd.to_numeric(df_fac[col], errors='coerce')

out = RAW_DIR / 'echo-facilities-iowa.csv'
df_fac.to_csv(out, index=False)
print(f'Saved {len(df_fac):,} facilities → {out}')
df_fac.head()

## 2. NAICS industry codes

In [ ]:
ia_ids = set(df_fac['npdes_id'].dropna())

text = stream_csv('NPDES_NAICS.csv')
reader = csv.DictReader(io.StringIO(text))
rows = [r for r in reader if r.get('NPDES_ID', '').strip().strip('"') in ia_ids]

df_naics = pd.DataFrame(rows)
df_naics.columns = [c.strip('"') for c in df_naics.columns]
df_naics.rename(columns={
    'NPDES_ID':              'npdes_id',
    'NAICS_CODE':            'naics_code',
    'NAICS_DESC':            'naics_desc',
    'PRIMARY_INDICATOR_FLAG': 'primary',
}, inplace=True)

out = RAW_DIR / 'echo-naics-iowa.csv'
df_naics.to_csv(out, index=False)
print(f'Saved {len(df_naics):,} NAICS rows → {out}')
df_naics[df_naics['primary'] == 'Y']['naics_desc'].value_counts().head(10)

## 3. SIC industry codes

In [ ]:
text = stream_csv('NPDES_SICS.csv')
reader = csv.DictReader(io.StringIO(text))
rows = [r for r in reader if r.get('NPDES_ID', '').strip().strip('"') in ia_ids]

df_sic = pd.DataFrame(rows)
df_sic.columns = [c.strip('"') for c in df_sic.columns]
df_sic.rename(columns={
    'NPDES_ID':              'npdes_id',
    'SIC_CODE':              'sic_code',
    'SIC_DESC':              'sic_desc',
    'PRIMARY_INDICATOR_FLAG': 'primary',
}, inplace=True)

out = RAW_DIR / 'echo-sics-iowa.csv'
df_sic.to_csv(out, index=False)
print(f'Saved {len(df_sic):,} SIC rows → {out}')
df_sic[df_sic['primary'] == 'Y']['sic_desc'].value_counts().head(10)

## 4. Summary and DMR cross-reference

In [ ]:
print('=== Facility type codes ===')
print(df_fac['facility_type_code'].value_counts().to_string())

print('\n=== Primary NAICS (top 10) ===')
print(df_naics[df_naics['primary']=='Y']['naics_desc'].value_counts().head(10).to_string())

# Cross-reference against DMR permit numbers
dmr_permits = set()
for f in sorted(RAW_DIR.glob('NPDES_DMRS_FY*.csv')):
    chunk = pd.read_csv(f, usecols=['EXTERNAL_PERMIT_NMBR'], dtype=str)
    dmr_permits.update(chunk['EXTERNAL_PERMIT_NMBR'].dropna().unique())

echo_ids = set(df_fac['npdes_id'].dropna())
print(f'\nDMR unique permit numbers:   {len(dmr_permits):,}')
print(f'ECHO Iowa facilities:         {len(echo_ids):,}')
print(f'Matched (DMR ∩ ECHO):         {len(dmr_permits & echo_ids):,}')
print(f'DMR permits not found in ECHO: {len(dmr_permits - echo_ids):,}')